# Lightweight Audio Source Separation - Demo

This notebook demonstrates:
- Audio source separation (like FL Studio stems)
- Frequency-based separation
- Spectrogram visualization
- Optimized for edge devices

In [ ]:
# Install required packages
!pip install librosa soundfile numpy matplotlib

In [ ]:
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import time

print("Libraries loaded successfully")

## 1. Generate Mixed Audio

In [ ]:
def generate_mixed_audio(filename="mixed.wav", sr=16000, duration=3):
    """Generate mixed audio with voice and background"""
    t = np.linspace(0, duration, int(sr * duration))
    
    # Voice (lower frequencies, speech-like)
    voice = 0.5 * np.sin(2 * np.pi * 300 * t)
    voice += 0.3 * np.sin(2 * np.pi * 500 * t)
    voice += 0.2 * np.sin(2 * np.pi * 700 * t)
    voice += 0.15 * np.sin(2 * np.pi * 900 * t)
    
    # Background (higher frequencies, noise-like)
    np.random.seed(42)
    background = 0.2 * np.sin(2 * np.pi * 2000 * t)
    background += 0.15 * np.sin(2 * np.pi * 3500 * t)
    background += 0.1 * np.random.randn(len(t))
    
    # Mix
    mixed = voice + background
    mixed = mixed / np.max(np.abs(mixed)) * 0.8
    
    sf.write(filename, mixed, sr)
    print(f"Created: {filename}")
    return filename

test_file = generate_mixed_audio()

## 2. Source Separation

In [ ]:
class FrequencySeparator:
    """Simple frequency-based source separation"""
    
    def __init__(self, split_frequency=1500):
        self.split_frequency = split_frequency
    
    def separate(self, audio, sr):
        """Separate voice and background"""
        # STFT
        stft = librosa.stft(audio, n_fft=512, hop_length=128)
        magnitude = np.abs(stft)
        phase = np.angle(stft)
        
        # Get frequency bins
        freq_bins = np.fft.fftfreq(512, 1/sr)
        split_idx = np.searchsorted(freq_bins, self.split_frequency)
        
        # Create masks
        voice_mask = np.ones_like(magnitude)
        bg_mask = np.ones_like(magnitude)
        
        # Attenuate high frequencies for voice
        voice_mask[split_idx:, :] = 0.1
        bg_mask[:split_idx, :] = 0.1
        
        # Apply masks
        voice_spec = magnitude * voice_mask
        bg_spec = magnitude * bg_mask
        
        # ISTFT
        voice_audio = librosa.istft(voice_spec * np.exp(1j * phase), hop_length=128)
        bg_audio = librosa.istft(bg_spec * np.exp(1j * phase), hop_length=128)
        
        return voice_audio, bg_audio

# Load audio
audio, sr = librosa.load(test_file, sr=16000)
print(f"Audio: {len(audio)/sr:.2f}s at {sr}Hz")

# Separate
separator = FrequencySeparator(split_frequency=1500)
voice, background = separator.separate(audio, sr)

print(f"Voice track: {len(voice)/sr:.2f}s")
print(f"Background: {len(background)/sr:.2f}s")

## 3. Visualize Separation

In [ ]:
# Plot all spectrograms
def get_spectrogram(audio):
    stft = librosa.stft(audio, n_fft=512, hop_length=128)
    return np.abs(stft)

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

for i, (data, title) in enumerate([
    (audio, 'MIXED Audio'),
    (voice, 'VOICE'),
    (background, 'BACKGROUND')
]):
    mag = get_spectrogram(data)
    db = 20 * np.log10(mag + 1e-10)
    
    plt.subplot(3, 1, i+1)
    plt.imshow(db, aspect='auto', origin='lower', cmap='viridis', vmin=-60, vmax=20)
    plt.title(title)
    plt.colorbar()

plt.tight_layout()
plt.show()

## 4. Save Separated Tracks

In [ ]:
# Save separated tracks
sf.write('voice_output.wav', voice, sr)
sf.write('background_output.wav', background, sr)
print("Saved: voice_output.wav, background_output.wav")

## 5. Performance Test

In [ ]:
# Test performance
def test_performance(audio, sr, num_runs=10):
    """Test separation speed"""
    
    start = time.time()
    for _ in range(num_runs):
        _ = separator.separate(audio, sr)
    elapsed = time.time() - start
    
    avg_time = (elapsed / num_runs) * 1000  # ms
    duration = len(audio) / sr
    
    print(f"=== Performance ===")
    print(f"Duration: {duration:.2f}s")
    print(f"Avg time: {avg_time:.2f}ms")
    print(f"Real-time factor: {duration / (avg_time/1000):.2f}x")
    print(f"CPU friendly: {'Yes' if avg_time < duration*1000 else 'No'}")

test_performance(audio, sr)

## 🎯 Summary

This notebook demonstrates:
- Source separation basics
- Frequency-based masking
- Spectrogram visualization
- Edge-friendly processing

For production, use Deep Learning models like:
- Demucs
- Spleeter
- Open-Unmix